# 🤖 Smart HR Recruitment — Multi-Agent System (Google ADK)

**Google AI Agents Capstone · Track: _Agents for Business_**

This notebook automates the **end-to-end recruitment pipeline** with four
specialized AI agents built on the **Google Agent Development Kit (ADK)**:

| # | Agent | Job |
|---|-------|-----|
| 1 | **JD Agent** | Analyze a job description → structured requirements (JSON) |
| 2 | **Resume Screener Agent** | Parse + score + rank resumes against the JD |
| 3 | **Interview Scheduler Agent** | Draft personalized invites + propose slots |
| 4 | **Insights Agent** | Aggregate the funnel → metrics + visual dashboard |

### Concepts demonstrated (capstone requires ≥ 3 — this covers 4)
1. **Multi-agent systems with ADK** — agents composed in a `SequentialAgent`; deterministic stage tools write full results to shared session state.
2. **MCP Server integration** — a Model Context Protocol server gives agents *sandboxed* file access.
3. **Agent Skills** — reusable tools: `resume_parser_skill`, `candidate_scorer_skill`, `email_drafter_skill`.
4. **Security / PII masking + RBAC** — emails/phones masked in logs; role-based access control.

> **Design note:** each agent is a real ADK `LlmAgent` with one deterministic stage tool.
> The same skills are callable directly, so the full pipeline runs deterministically
> **with or without a Gemini API key** — ideal for reproducible grading on Kaggle.

## 🏗️ Architecture

```
                         ┌──────────────────────────────────────────┐
                         │          SequentialAgent (ADK)            │
                         │     smart_hr_recruitment_pipeline         │
                         └──────────────────────────────────────────┘
   job_description ─▶ [JD Agent] ─jd_requirements▶ [Resume Screener]
                                                         │ ranked_candidates
                                                         ▼
                          insights_report ◀─ [Insights] ◀─ [Scheduler]
                                                         interview_emails

   Cross-cutting:  ├─ MCP Filesystem Server (read resumes / save reports)
                   └─ Security layer (PII masking in logs + RBAC)

   Stage tools (FunctionTools): adk_jd_stage · adk_screening_stage · adk_scheduler_stage · adk_insights_stage
```
Each agent calls one stage tool. Full results live in session state; the model sees
only small, PII-free summaries.

## 1. Environment setup
Install the Google ADK and supporting libraries.

In [ ]:
# On Kaggle, uncomment to install. Versions are pinned in requirements.txt.
# !pip install -q google-adk==2.3.0 google-genai==2.10.0 mcp==1.28.1 pandas==3.0.3 matplotlib==3.11.0 pypdf==6.14.2

import os, sys
from pathlib import Path
print("Python:", sys.version.split()[0])

# Locate the repository root containing both src/ and data/.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / "src").is_dir() and (REPO_ROOT / "data").is_dir()):
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").is_dir():
    raise FileNotFoundError("Could not locate the repository root containing src/. Run the notebook from inside the repo.")
sys.path.insert(0, str(REPO_ROOT))
os.environ["HR_DATA_ROOT"] = str(REPO_ROOT / "data")
os.chdir(REPO_ROOT)
print("Repository root:", REPO_ROOT)

try:
    import google.adk  # noqa
    print("google-adk available")
    ADK_INSTALLED = True
except Exception:
    print("google-adk not installed - pipeline will use the deterministic skill runner.")
    ADK_INSTALLED = False


### (Optional) Gemini API key
ADK's `LlmAgent`s call Gemini. On Kaggle, add your key via **Add-ons → Secrets**
(`GOOGLE_API_KEY`). Without it, the notebook automatically falls back to the
deterministic skill runner — every cell below still produces real output.

In [ ]:
# Wire up the API key from Kaggle Secrets if present (safe no-op otherwise).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("Gemini API key loaded from Kaggle Secrets ✓")
except Exception:
    print("No Gemini key found — using deterministic skill runner (fully functional).")

## 2. Dataset

The repository ships **18 fully synthetic resumes** in `data/resumes/` (15 text
+ 3 PDF) and one job description. Every person, employer, email, and phone
number is invented; emails use `@example.com` and phone numbers use the
reserved `555` exchange. The set deliberately covers:

- a fresh graduate and a junior developer
- a candidate with a career gap
- a keyword stuffer
- a non-English name and non-English resume content
- an image-only **scanned-style PDF** (no text layer)
- a PDF with **hidden white text**
- a resume carrying an **"ignore previous instructions"** prompt injection

Generate or refresh the fixtures with `python tools/generate_synthetic_resumes.py`.


In [ ]:
from pathlib import Path

RESUME_DIR = REPO_ROOT / "data" / "resumes"
JD_DIR = REPO_ROOT / "data" / "job_descriptions"
OUTPUT_DIR = REPO_ROOT / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

resume_files = sorted(p.name for p in RESUME_DIR.iterdir() if p.is_file())
assert len(resume_files) >= 16, f"Expected >=16 synthetic resumes, found {len(resume_files)}"
print(f"Using {len(resume_files)} synthetic resumes from {RESUME_DIR}:")
for name in resume_files:
    print("  -", name)


## 3. Security layer — PII masking + RBAC

Written first because it's **cross-cutting**: every other component imports it.
- `mask_pii` / `PiiMaskingFilter` — emails, phones, URLs are replaced with typed
  placeholders before anything is logged.
- `SecurityContext` + `Role` — role-based access control gates every sensitive
  action (only `recruiter`/`admin` may view unmasked PII or send invitations).

**Import the security module from `src/`** (no inline source copy).

In [ ]:
import src.security as sec
from src.security import SecurityContext, Role, AccessDeniedError
print("Loaded security module from:", sec.__file__)


In [ ]:
sample = "Contact Priya at priya.menon@example.com or +1 (415) 555-0103 (linkedin.com/in/priya)"
print("RAW    :", sample)
print("MASKED :", sec.mask_pii(sample))

print("\nRBAC checks:")
print("  recruiter can view_unmasked_pii :", SecurityContext(role=Role.RECRUITER).can('view_unmasked_pii'))
print("  viewer    can view_unmasked_pii :", SecurityContext(role=Role.VIEWER).can('view_unmasked_pii'))
try:
    SecurityContext(role=Role.VIEWER).authorize('read_resume')
except AccessDeniedError as e:
    print("  viewer read_resume ->", e)


## 4. Agent Skills (reusable tools)

These are the **reusable Agent Skills** the capstone asks for. Each is a pure,
testable function that ADK agents invoke as a `FunctionTool`:
`resume_parser_skill`, `jd_parser_skill`, `candidate_scorer_skill`,
`email_drafter_skill` (+ `suggest_interview_slots`). A normalized **skill
vocabulary** makes matching robust (`k8s`≈`kubernetes`, `postgres`≈`postgresql`).

**Import the reusable Agent Skills from `src/`.**

In [ ]:
import src.skills as sk
print("Loaded skills module from:", sk.__file__)


In [ ]:
demo = sk.resume_parser_skill(
    (REPO_ROOT / "data" / "resumes" / "resume_senior_backend_daniel_okafor.txt").read_text(encoding="utf-8"),
    "daniel",
)
print("resume_parser_skill ->")
for k, v in demo.items():
    print(f"  {k}: {v}")


## 5. MCP Server — sandboxed filesystem access

Rather than handing agents raw disk access, we expose a small, audited tool
surface over the **Model Context Protocol**: `list_resumes`, `read_resume`,
`read_job_description`, `save_report` — with path-traversal protection. ADK
agents connect via `MCPToolset`; the same functions are importable directly.

**Import the sandboxed MCP filesystem module from `src/`.**

In [ ]:
import src.mcp_filesystem_server as fs
print("Loaded MCP filesystem module from:", fs.__file__)


In [ ]:
import importlib, src.mcp_filesystem_server as fs; importlib.reload(fs)
print("MCP tool list_resumes() →", fs.list_resumes())
print("MCP tool read_job_description() → first 80 chars:")
print("  ", fs.read_job_description()[:80].replace(chr(10),' '))

### PDF parsing spotlight

`read_resume` transparently handles **both text and PDF** resumes via `pypdf`.
The synthetic dataset includes three PDFs to exercise the edge cases:

- `resume_pdf_sofia_martinez.pdf` - a normal text PDF
- `resume_scanned_ravi_kumar.pdf` - an image-only **scanned-style PDF** (no text layer)
- `resume_hidden_white_text_emma_jones.pdf` - visible text plus **white-on-white hidden text**
  containing a prompt-injection payload


In [ ]:
%pip install pypdf --quiet

In [ ]:
normal_pdf = "resume_pdf_sofia_martinez.pdf"
scanned_pdf = "resume_scanned_ravi_kumar.pdf"
hidden_pdf = "resume_hidden_white_text_emma_jones.pdf"

for pdf_name in (normal_pdf, scanned_pdf, hidden_pdf):
    pdf_text = fs.read_resume(pdf_name)
    print(f"\n--- {pdf_name}: {len(pdf_text)} extracted chars ---")
    print(pdf_text[:200].replace("\n", " "))

parsed_pdf = sk.resume_parser_skill(fs.read_resume(normal_pdf), source_id=normal_pdf)
print("\nParsed normal PDF candidate ->")
for k, v in parsed_pdf.items():
    print(f"  {k}: {v}")

from src.sanitize import sanitize_untrusted_text
hidden_text = fs.read_resume(hidden_pdf)
print("\nHidden text after sanitisation (first 400 chars):")
print(sanitize_untrusted_text(hidden_text)[:400])


## 6. The four agents (Google ADK `LlmAgent`s)

Each agent is a Gemini-backed `LlmAgent` with exactly one deterministic stage
tool from `src/adk_stages.py`. The tool stores full results in shared session
state and returns a small, PII-free summary to the model. `build_recruitment_pipeline()`
composes the four agents into a `SequentialAgent` — **this is the multi-agent orchestration.**

**Import the ADK agents module from `src/`.**

In [ ]:
import src.agents as agents
print("Loaded agents module from:", agents.__file__)
print("ADK available in this kernel:", agents.ADK_AVAILABLE)


In [ ]:
import importlib, src.agents as agents; importlib.reload(agents)
print("ADK available in this kernel:", agents.ADK_AVAILABLE)
if agents.ADK_AVAILABLE:
    pipeline = agents.build_recruitment_pipeline()
    print("Built SequentialAgent:", pipeline.name)
    print("Sub-agents:", [a.name for a in pipeline.sub_agents])
else:
    print("ADK not installed — the deterministic orchestrator (next section) runs the same skills.")

### Running the ADK pipeline with a `Runner` (when a Gemini key is present)

This is how ADK actually executes the multi-agent graph: a `Runner` drives the
`SequentialAgent` over a session, and each sub-agent writes to `output_key`.

In [ ]:
async def run_with_adk(jd_filename=""):
    import src.orchestrator as orch
    return await orch.run_adk_pipeline_async(role="recruiter", jd_filename=jd_filename)

if agents.ADK_AVAILABLE and os.environ.get("GOOGLE_API_KEY"):
    import asyncio
    result = asyncio.run(run_with_adk())
    print("ADK mode          :", result["mode"])
    print("Applicants        :", result["insights"]["total_applicants"])
    print("Shortlisted       :", result["insights"]["shortlisted"])
    print("Report saved to   :", result["insights"]["report_path"])
else:
    print("Skipping live ADK run (no Gemini key). The deterministic run below produces full output.")


## 7. Deterministic orchestrator (always-runnable mirror)

`orchestrator.py` threads the four stages exactly like the ADK `SequentialAgent`
(`jd_requirements → ranked_candidates → interview_emails → insights_report`),
applying RBAC + PII masking at every stage. We use it below so every demo cell
yields real output regardless of API-key availability.

**Import the deterministic orchestrator from `src/`.**

In [ ]:
import src.orchestrator as orch
print("Loaded orchestrator module from:", orch.__file__)


## 8. Demo — Agent 1: JD Agent

In [ ]:
import importlib, src.orchestrator as orch; importlib.reload(orch)
from src.security import SecurityContext, Role
ctx = SecurityContext(user="demo_recruiter", role=Role.RECRUITER)

requirements = orch.run_jd_agent(ctx)
import json; print(json.dumps(requirements, indent=2))

## 9. Demo — Agent 2: Resume Screener Agent
Parses all resumes, scores each against the JD, returns a ranked list with match %.

In [ ]:
%pip install pandas matplotlib pypdf --quiet

In [ ]:
ranked = orch.run_resume_screener(ctx, requirements)
import pandas as pd
df = pd.DataFrame(ranked)[["rank","name","match_percent","experience_years","recommendation","missing_required"]]
df

## 10. Demo — Agent 3: Interview Scheduler Agent
Drafts personalized invites + proposes slots for shortlisted candidates (match ≥ 75%).

In [ ]:
emails = orch.run_scheduler(ctx, ranked, role_title=requirements["title"])
print(f"{len(emails)} invitation(s) drafted.\n")
print("="*70)
print(emails[0]["subject"])
print("="*70)
print(emails[0]["body"])

## 11. Demo — Agent 4: Insights Agent
Aggregates the funnel into metrics, saves a markdown report, and powers the dashboard.

In [ ]:
insights = orch.run_insights(ctx, requirements, ranked, emails)
print(json.dumps({k:v for k,v in insights.items() if k not in ('score_distribution','candidate_labels')}, indent=2, default=str))

### 📊 Visual dashboard

In [ ]:
import matplotlib.pyplot as plt
from src.config import SHORTLIST_THRESHOLD, REVIEW_THRESHOLD
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
fig.suptitle(f"Recruitment Dashboard — {insights['role']}", fontsize=14, fontweight="bold")

# (a) Candidate match scores
labels = insights["candidate_labels"]; scores = insights["score_distribution"]
colors = ["#2e7d32" if s>=SHORTLIST_THRESHOLD else "#f9a825" if s>=REVIEW_THRESHOLD else "#c62828" for s in scores]
axes[0].barh(labels[::-1], scores[::-1], color=colors[::-1])
axes[0].axvline(SHORTLIST_THRESHOLD, ls="--", color="gray"); axes[0].set_title("Candidate Match %")
axes[0].set_xlabel("match %"); axes[0].set_xlim(0,100)

# (b) Funnel
funnel_vals = [insights["total_applicants"], insights["shortlisted"]]
axes[1].bar(["Applicants","Shortlisted"], funnel_vals, color=["#1565c0","#2e7d32"])
axes[1].set_title(f"Funnel (shortlist rate {insights['shortlist_rate_percent']}%)")
for i,v in enumerate(funnel_vals): axes[1].text(i, v+0.05, str(v), ha="center")

# (c) Top skills
sk_names = [s for s,_ in insights["top_skills_found"]]
sk_counts = [c for _,c in insights["top_skills_found"]]
axes[2].barh(sk_names[::-1], sk_counts[::-1], color="#6a1b9a")
axes[2].set_title("Top Skills in Applicant Pool"); axes[2].set_xlabel("count")

plt.tight_layout(); plt.savefig("data/output/dashboard.png", dpi=110, bbox_inches="tight")
plt.show()
print("Dashboard saved to data/output/dashboard.png")

## 12. 🚀 Full pipeline — end-to-end run

One call runs all four agents in sequence under a single `SecurityContext`,
with MCP file I/O and PII-masked logging throughout. This is the capstone's
**multi-agent orchestration demo**.

In [ ]:
result = orch.run_pipeline(role="recruiter")

print("\n" + "="*72)
print("PIPELINE SUMMARY")
print("="*72)
r = result["insights"]
print(f"Role              : {r['role']}")
print(f"Applicants        : {r['total_applicants']}")
print(f"Shortlisted       : {r['shortlisted']}  (rate {r['shortlist_rate_percent']}%)")
print(f"Avg match score   : {r['average_match_percent']}%")
print(f"Invites drafted   : {len(result['interview_emails'])}")
print(f"Report saved to   : {r['report_path']}")
print("\nTop 3 candidates:")
for c in result["ranked_candidates"][:3]:
    print(f"  #{c['rank']} {c['name']:16s} {c['match_percent']:5.1f}%  — {c['recommendation']}")

### Audit trail (RBAC decisions)
Every gated action is recorded — useful for compliance.

In [ ]:
import pandas as pd
pd.DataFrame(result["audit_log"])

## 13. Security spotlight — same data, different roles

The same pipeline run by a `viewer` cannot read resumes or send invites, and
sees PII masked. This demonstrates RBAC + PII masking working together.

In [ ]:
from src.security import AccessDeniedError
try:
    orch.run_pipeline(role="viewer")
except AccessDeniedError as e:
    print("Viewer correctly blocked:", e)

sample = "Email priya.menon@example.com, phone +1 (415) 555-0103"
print("\nrecruiter view:", SecurityContext(role=Role.RECRUITER).maybe_mask(sample))
print("viewer view   :", SecurityContext(role=Role.VIEWER).maybe_mask(sample))


## 14. Generated report (saved via the MCP `save_report` tool)

In [ ]:
print(Path(result["insights"]["report_path"]).read_text(encoding="utf-8"))


## Conclusion

We built a working **Smart HR Recruitment multi-agent system** on Google ADK that:

- **Orchestrates four specialized agents** (JD -> Screener -> Scheduler -> Insights)
  via an ADK `SequentialAgent`; deterministic stage tools share full results through
  session state and keep ranking/email decisions auditable.
- Exposes **sandboxed file access through an MCP server**.
- Reuses **Agent Skills** (`resume_parser`, `candidate_scorer`, `email_drafter`)
  as `FunctionTool`s.
- Enforces a **security layer**: PII masking in logs + role-based access control.

The pipeline takes a raw JD and the 18 fully synthetic resumes and produces a
ranked shortlist, personalised interview invitations, and a visual
hiring-insights dashboard. All modules are imported from `src/` - the notebook
no longer writes copies of the source or a duplicate dataset.

*Built for the Google AI Agents Capstone - "Agents for Business" track.*
